# 02 — Build and Run an Agent

This notebook walks you through interacting with a deployed agent or creating one from scratch.
Run cells top-to-bottom. Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [ ]:
# Change this to work with a different agent
AGENT_NAME = "code-helper"

## Install Dependencies

In [ ]:
%pip install azure-ai-agents azure-identity pydantic-settings python-dotenv -q
# Or if using uv from a terminal: uv sync

## Connect to Foundry

In [ ]:
from dotenv import load_dotenv
import os
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

# Load connection string from .env
load_dotenv()
CONNECTION_STRING = os.environ["FOUNDRY_PROJECT_CONNECTION_STRING"]

client = AgentsClient(
    endpoint=CONNECTION_STRING,
    credential=DefaultAzureCredential()
)
print("✓ Connected to Azure AI Foundry")

## Step 1: Get or Deploy the Agent

Uses the existing deployed agent if found, otherwise deploys it automatically via the registry.

In [ ]:
from agents.registry import REGISTRY

# Check if agent is already deployed
agent = None
for a in client.list_agents():
    if a.name == AGENT_NAME:
        agent = a
        break

if agent:
    print(f"✓ Found existing agent: {agent.name} (id: {agent.id}, model: {agent.model})")
else:
    # Deploy via the registry
    print(f"Agent '{AGENT_NAME}' not found — deploying...")
    entry = REGISTRY.get_agent(AGENT_NAME)
    config = entry.config_class()
    agent = entry.factory(config)
    print(f"✓ Deployed new agent: {agent.name} (id: {agent.id}, model: {agent.model})")

## Step 2: Create a Thread and Send a Message

In [ ]:
# Create a conversation thread
thread = client.threads.create()
print(f"✓ Thread created: {thread.id}")

# Post a message — change the content to test different prompts
message = client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Hello! Can you greet a user named Alice?",
)
print(f"✓ Message posted: {message.id}")

## Step 3: Run the Agent and Get a Response

In [ ]:
import importlib
from azure.ai.agents.models import MessageRole

# Load and register the agent's tool functions so the SDK can auto-execute them
module_name = AGENT_NAME.replace("-", "_")
tools_module = importlib.import_module(f"agents.{module_name}.tools")
if hasattr(tools_module, "TOOLS") and tools_module.TOOLS:
    for tool in tools_module.TOOLS:
        client.enable_auto_function_calls(tool)
    print(f"✓ Registered {len(tools_module.TOOLS)} tool(s) for auto-execution")

# Create and process the run (SDK handles polling + tool execution)
run = client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)

print(f"Run status: {run.status}")

# Retrieve the response
last_msg = client.messages.get_last_message_text_by_role(
    thread_id=thread.id,
    role=MessageRole.AGENT,
)

response = last_msg.text.value if last_msg else "(no response)"
print(f"\nAgent response:\n{response}")

## Step 4: Cleanup

Delete the thread. The deployed agent is kept for future use.

In [ ]:
# Clean up the thread (keeps the deployed agent)
client.threads.delete(thread.id)
print("✓ Thread deleted (agent kept for future use)")

## Next Steps

- Change the message content in Step 2 to test different prompts
- Try a different agent: set `AGENT_NAME = "doc-assistant"` and re-run
- Add custom tools — see the [Custom Tools Guide](../docs/custom-tools-guide.md) for a full walkthrough
- See **03_scaffold_agent.ipynb** to create a brand new agent